# Lesson 27 Lab — CUTLASS, cuBLAS, cuDNN, or Triton?

**Puzzle:** When library coverage, epilogue fusion, maintainability, and peak tuning change together, which observation tells you whether the kernel, layout, toolchain, or hardware boundary is responsible?

This notebook retains one complete RTX 5090 execution.


## Why this matters

This lab isolates library coverage, epilogue fusion, maintainability, and peak tuning and keeps its comparison path explicit.


## 0. Predict before running

Predict correctness, warm latency ordering, and the first boundary case. Write what would disprove each prediction.


## 1. Theory and mechanism

cuBLAS and cuDNN package highly tuned standard operations; CUTLASS provides C++ templates for architecture-aware kernels; Triton offers a compact DSL for custom blocked dataflow. The decision depends on operation coverage, fusion value, platform range, peak target, and maintenance budget.


## 2. Trace the mechanism

```mermaid
flowchart LR
  A["Frozen input + contract"] --> B["library coverage, epilogue fusion, maintainability, and peak tuning"]
  B --> C["Triton candidate"]
  B --> D["CUDA / library control"]
  C --> E["correctness + samples"]
  D --> E
  E --> F["bounded decision"]
```


## 3. Inspect the comparison boundary

Baseline: named PyTorch CUDA/library or standard-grid path. Candidate: reviewed Triton kernel or explicit model described below.

A custom GEMM that wins one shape but loses library coverage, testing, and future architectures may be a net regression.


## 4. Inspect the execution environment

The next cell asserts CUDA and records GPU, target, PyTorch, CUDA runtime, Triton, Python, and seed.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT / "scripts"))
from chapter05_runtime import environment, run_lesson

LESSON_NO = 27
LESSON_TITLE = 'CUTLASS, cuBLAS, cuDNN, or Triton?'
ENV = environment(LESSON_NO)
print(json.dumps(ENV, indent=2, ensure_ascii=False))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "triton": "3.7.1",
  "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
  "python": "3.12.3",
  "seed": 20260840
}


## 5. Freeze the experiment

**Experiment:** Fuse ReLU into a teaching Triton GEMM store and compare with torch.mm plus ReLU.

Inputs, output contract, timer, and target stay fixed across compared paths.


## 6. Inspect and execute the reviewed code

The next cell calls the shared reviewed kernel source, retains full samples in `metrics`, checks maximum error, and prints the bounded analysis.


In [2]:
metrics, analysis_en, analysis_zh = run_lesson(LESSON_NO)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print(analysis_en)


{
  "primary": 0.022911999374628067,
  "secondary": 0.019360000267624855,
  "max_abs_error": 0.0,
  "passed": true,
  "details": {
    "triton_tflops": 11.715933280674575,
    "library_tflops": 13.865467576924393,
    "dtype": "torch.float16",
    "fused_relu": true,
    "triton_samples_ms": [
      0.03846399858593941,
      0.027904000133275986,
      0.02457600086927414,
      0.02316799946129322,
      0.023455999791622162,
      0.025567999109625816,
      0.023552000522613525,
      0.022911999374628067,
      0.022048000246286392,
      0.02239999920129776,
      0.02147199958562851,
      0.02175999991595745,
      0.022463999688625336,
      0.022463999688625336,
      0.020128000527620316
    ],
    "library_samples_ms": [
      0.020800000056624413,
      0.01974399946630001,
      0.019711999222636223,
      0.018912000581622124,
      0.0191040001809597,
      0.019519999623298645,
      0.019231999292969704,
      0.019200000911951065,
      0.018848000094294548,
      0.

## 7. Read the retained RTX 5090 result

**Environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Triton 3.7.1; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Fused Triton median | 0.0229 ms |
| Library plus ReLU median | 0.0194 ms |
| Maximum absolute error | 0.000e+00 |
| Acceptance gate | true |


## 8. Explain without overclaiming

The custom path fused ReLU into the GEMM store and took 0.0229 ms; torch.mm plus ReLU took 0.0194 ms. This is one shape, not a library ranking.

A named Triton or PyTorch CUDA path executed on the recorded GPU. The result applies to the printed shape, dtype, implementation, and software stack; internal hardware causes require profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, full metrics, bilingual analysis, evidence label, and bounded conclusion.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": LESSON_NO,
    "title": LESSON_TITLE,
    "environment": ENV,
    "evidence_label": 'native-backend',
    "metrics": metrics,
    "analysis_en": analysis_en,
    "analysis_zh": analysis_zh,
    "conclusion": 'Start from the strongest covered library; own a custom kernel only when the uncovered constraint has measured product value.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 27,
  "title": "CUTLASS, cuBLAS, cuDNN, or Triton?",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "triton": "3.7.1",
    "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
    "python": "3.12.3",
    "seed": 20260840
  },
  "evidence_label": "native-backend",
  "metrics": {
    "primary": 0.022911999374628067,
    "secondary": 0.019360000267624855,
    "max_abs_error": 0.0,
    "passed": true,
    "details": {
      "triton_tflops": 11.715933280674575,
      "library_tflops": 13.865467576924393,
      "dtype": "torch.float16",
      "fused_relu": true,
      "triton_samples_ms": [
        0.03846399858593941,
        0.027904000133275986,
        0.02457600086927414,
        0.02316799946129322,
        0.023455999791622162,
        0.025567999109625816,
        0.023552000522613525,
        0.022911999374628067,
        0.022048000246286392,
     

## 10. Make the bounded decision

> Start from the strongest covered library; own a custom kernel only when the uncovered constraint has measured product value.

**Failure analysis:** A custom GEMM that wins one shape but loses library coverage, testing, and future architectures may be a net regression.


## 11. Extend and review

Add an awkward shape and non-contiguous layout. Stop on correctness failure. See `README.md` for references and the full review checklist.
